<a href="https://colab.research.google.com/github/AvishkaPrabudi/AI-Powered-NLP-Chatbot/blob/main/AI_Powered_NLP_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch fastapi uvicorn scikit-learn

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.preprocessing import LabelEncoder

In [ ]:
texts = [
    "book an appointment",
    "cancel my booking",
    "hello",
    "hi there",
    "schedule doctor visit",
    "delete appointment"
]

labels = [
    "book",
    "cancel",
    "greeting",
    "greeting",
    "book",
    "cancel"
]

In [ ]:
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

print(le.classes_)

['book' 'cancel' 'greeting']


In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(set(labels))
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

labels_tensor = torch.tensor(labels_encoded)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

model.train()

for epoch in range(3):
    optimizer.zero_grad()

    outputs = model(**inputs, labels=labels_tensor)
    loss = outputs.loss

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Epoch 1, Loss: 1.0526643991470337
Epoch 2, Loss: 0.8131076693534851
Epoch 3, Loss: 0.7152724862098694


In [ ]:
def predict_intent(text):
    model.eval()

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    outputs = model(**inputs)
    logits = outputs.logits

    predicted_class_id = torch.argmax(logits, dim=1).item()

    return le.inverse_transform([predicted_class_id])[0]

In [ ]:
test_inputs = [
    "hello",
    "book appointment",
    "cancel booking"
]

for user_input in test_inputs:
    print("You:", user_input)

    intent = predict_intent(user_input)

    if intent == "book":
        print("Bot: Sure, I can help you book an appointment!")
    elif intent == "cancel":
        print("Bot: Your appointment has been cancelled.")
    else:
        print("Bot: Hello! How can I help you?")

You: hello
Bot: Hello! How can I help you?
You: book appointment
Bot: Sure, I can help you book an appointment!
You: cancel booking
Bot: Your appointment has been cancelled.


In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    return {"message": "Chatbot API running"}

@app.get("/chat")
def chat(user_input: str):
    intent = predict_intent(user_input)
    return {"intent": intent}

In [ ]:
import re

def extract_entities(text):
    entities = {}

    # Date detect karanna (simple)
    if "tomorrow" in text.lower():
        entities["DATE"] = "tomorrow"

    # Numbers (dates wage)
    numbers = re.findall(r'\d+', text)
    if numbers:
        entities["NUMBER"] = numbers

    # Keywords
    if "doctor" in text.lower():
        entities["SERVICE"] = "doctor"

    return entities

In [ ]:
context_memory = []

def update_context(user_input):
    context_memory.append(user_input)

    # last 3 messages witharak keep karanawa
    return context_memory[-3:]

In [ ]:
def generate_response(user_input, intent, entities, context):

    prompt = f"""
    Context: {context}
    User: {user_input}
    Intent: {intent}
    Entities: {entities}
    """

    # simple intelligent responses
    if intent == "book":
        return f"I can help you book a {entities.get('SERVICE', 'service')} appointment on {entities.get('DATE', 'a chosen date')}."

    elif intent == "cancel":
        return "Your appointment has been cancelled successfully."

    else:
        return "Hello! How can I assist you today?"

In [ ]:
test_inputs = [
    "hello",
    "book doctor appointment tomorrow",
    "cancel my booking"
]

for user_input in test_inputs:
    print("You:", user_input)

    intent = predict_intent(user_input)
    entities = extract_entities(user_input)
    context = update_context(user_input)

    response = generate_response(user_input, intent, entities, context)

    print("Bot:", response)
    print("-" * 30)

You: hello
Bot: Hello! How can I assist you today?
------------------------------
You: book doctor appointment tomorrow
Bot: I can help you book a doctor appointment on tomorrow.
------------------------------
You: cancel my booking
Bot: I can help you book a service appointment on a chosen date.
------------------------------


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# test data
test_texts = [
    "book appointment",
    "cancel booking",
    "hello"
]

true_labels = ["book", "cancel", "greeting"]

predictions = [predict_intent(t) for t in test_texts]

accuracy = accuracy_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions, average="weighted")

print("Accuracy:", accuracy)
print("F1 Score:", f1)

Accuracy: 1.0
F1 Score: 1.0


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# test data
test_texts = [
    "book appointment",
    "cancel booking",
    "hello"
]

true_labels = ["book", "cancel", "greeting"]

predictions = [predict_intent(t) for t in test_texts]

accuracy = accuracy_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions, average="weighted")

print("Accuracy:", accuracy)
print("F1 Score:", f1)

Accuracy: 1.0
F1 Score: 1.0


In [ ]:
!pip install nest-asyncio pyngrok

In [ ]:
!pip install uvicorn

In [ ]:
import uvicorn

In [ ]:
!pip install fastapi uvicorn nest-asyncio pyngrok

In [ ]:
from fastapi import FastAPI
import uvicorn
import nest_asyncio
from pyngrok import ngrok

In [ ]:
app = FastAPI()

@app.get("/")
def home():
    return {"message": "Chatbot API running"}

@app.get("/chat")
def chat(user_input: str):
    # example simple response
    return {"intent": "book", "response": f"You said: {user_input}"}

In [ ]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

def run_app():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_app, daemon=True).start()

In [ ]:
nest_asyncio.apply()
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://ameer-scrappy-patronisingly.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
!pip install pyngrok

/usr/lib/python3.12/pathlib.py:404: RuntimeWarning: coroutine 'Server.serve' was never awaited
  parsed = [sys.intern(str(x)) for x in rel.split(sep) if x and x != '.']


In [ ]:
!ngrok config add-authtoken 3BLpNmBBIayKH9W8nuDesm8sscw_6NhqtiSxxyk9vtvZdcp17

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
import nest_asyncio
from pyngrok import ngrok

nest_asyncio.apply()
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://ameer-scrappy-patronisingly.ngrok-free.dev" -> "http://localhost:8000"
